# Importación de librerías

In [ ]:
# Módulo
import plotly.express as px
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Para que se puedan utilizar funciones desde el notebook
from src.utils.files import read_file
from src.utils.config import new_data_popularity,popularity, load_env_file
load_env_file()

# Carga de datos

In [ ]:
use_minio = False
minio = {"minio_write": False, "minio_read": use_minio}

In [ ]:
df_datos = read_file(popularity, minio)
df_nuevo = read_file(new_data_popularity, minio)

## Distribución de la variable respuesta

In [ ]:
fig = make_subplots(rows=1,cols=2, subplot_titles= ("Datos existentes","Nuevos datos extraídos"))

max_val = max(
    df_datos['recomendaciones_totales'].max(),
    df_nuevo['recomendaciones_totales'].max()
)
bin_size_nuevo = 10 
bin_size_datos = bin_size_nuevo*len(df_datos) / len(df_nuevo)


fig.add_trace(
    go.Histogram(
        x=df_datos['recomendaciones_totales'],
        marker_color="#A8754C",
        name='Recomendaciones',
        hovertemplate='<b>Recomendaciones:</b> %{x}<br><b>Frecuencia:</b> %{y}<extra></extra>',
        xbins=dict(size= bin_size_datos)
    ), row=1,col=1
)

fig.add_trace(
    go.Histogram(
        x=df_nuevo['recomendaciones_totales'],
        marker_color="#409ada",
        name='Recomendaciones',
        hovertemplate='<b>Recomendaciones:</b> %{x}<br><b>Frecuencia:</b> %{y}<extra></extra>',
        xbins=dict(size= bin_size_nuevo)
    ), row=1,col=2
)

fig.update_yaxes(title={"text": "Cantidad (Logarítmica)"}, type="log", row=1, col=1)
fig.update_yaxes(title={"text": "Cantidad (Logarítmica)"}, type="log", row=1, col=2)

fig.update_xaxes(title={"text": "Número de Recomendaciones Totales"}, row=1, col=1)
fig.update_xaxes(title={"text": "Número de Recomendaciones Totales"}, row=1, col=2)


fig.update_layout(
    title={
        'text': 'Distribución de Recomendaciones Totales en Nuevos Datos',
        'x': 0.5,
        'font': dict(size=30, weight="bold")
    },
    plot_bgcolor='whitesmoke',
    showlegend=False
)

fig.show()

In [ ]:
display(df_nuevo.sort_values(by='recomendaciones_totales', ascending=False).head(20).reset_index(drop=True))

## Comparación de distribución de las variables más importantes

In [ ]:
df_cleaned_new = df_nuevo.copy()
df_cleaned_new["viewCountTotal"] = df_cleaned_new.filter(like="video_statistics.viewCount").sum(axis=1)
df_cleaned_new["likeCountTotal"] = df_cleaned_new.filter(like="video_statistics.likeCount").sum(axis=1)
df_cleaned_new["commentCountTotal"] = df_cleaned_new.filter(like="video_statistics.commentCount").sum(axis=1)

### Correlaciones

In [ ]:
df_cleaned = df_datos.copy()
df_cleaned["viewCountTotal"] = df_cleaned.filter(like="video_statistics.viewCount").sum(axis=1)
df_cleaned["likeCountTotal"] = df_cleaned.filter(like="video_statistics.likeCount").sum(axis=1)
df_cleaned["commentCountTotal"] = df_cleaned.filter(like="video_statistics.commentCount").sum(axis=1)

numeric_columns = ["recomendaciones_totales", "description_len", "price_overview", "num_languages", "num_juegos_previos_developers", "ema_reviews_developers", "max_historico_reviews_developers", \
            "num_juegos_previos_publishers", "ema_reviews_publishers", "max_historico_reviews_publishers", "brillo", "viewCountTotal", \
            "likeCountTotal", "commentCountTotal"]

corr = df_cleaned[numeric_columns].corr(method='spearman')
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap='coolwarm', annot=True)
plt.show()

Las 3 variables con mayor correlación con la variable respuesta son: `num_languages` (0.28), `commentCountTotal` (0.27), `likeCountTotal` (0.26).

In [ ]:
columnas_mayor_correlacion = [
    "num_languages",
    "commentCountTotal",
    "likeCountTotal"
]

color_1 = "#A8754C"  # datos existentes
color_2 = "#187FF5"  # datos nuevos

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=columnas_mayor_correlacion
)

for i, col in enumerate(columnas_mayor_correlacion, start=1):

    fig.add_trace(
        go.Box(
            y=df_cleaned[col],
            name="Datos existentes",
            marker_color=color_1,
            boxpoints="outliers"
        ),
        row=1, col=i
    )

    fig.add_trace(
        go.Box(
            y=df_cleaned_new[col],
            name="Datos nuevos",
            marker_color=color_2,
            boxpoints="outliers"
        ),
        row=1, col=i
    )

fig.update_layout(
    title=dict(
        text="Comparación de distribuciones (Boxplots)",
        x=0.5,
        font=dict(size=22)
    ),
    template="plotly_white",
    showlegend=False,
    height=500,
    width=1000
)

fig.show()

Las distribuciones de las variables son muy similares, con muy pocas diferencias entre ellas exceptuando el número de juegos en ambos datasets. Es por ello que en los datos existentes hay más outliers en total.